In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split 
import tensorflow as tf
import tensorflow.keras as K
from tensorflow.keras import models
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score
import os
import json
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from pymongo import MongoClient
from dateutil import parser
from datetime import datetime, timedelta
# import lstm, time #도움을 주는 라이브러리들

In [2]:
conn = MongoClient("mongodb://203.249.90.5", 31010)
db = conn.get_database("traffic_Sejong_road")

In [3]:
os.environ['TF_CONFIG'] = json.dumps({
    'cluster': {
        'worker': ["10.244.1.86:8899",
                   "10.244.3.64:8899"]
    },
    'task': {'type': 'worker', 'index': 1}
})
multiworker_strategy = tf.distribute.experimental.MultiWorkerMirroredStrategy(tf.distribute.experimental.CollectiveCommunication.RING)

INFO:tensorflow:Enabled multi-worker collective ops with available devices: ['/job:worker/replica:0/task:1/device:CPU:0', '/job:worker/replica:0/task:1/device:XLA_CPU:0', '/job:worker/replica:0/task:1/device:XLA_GPU:0', '/job:worker/replica:0/task:1/device:GPU:0']
INFO:tensorflow:Using MirroredStrategy with devices ('/job:worker/task:1/device:GPU:0',)
INFO:tensorflow:MultiWorkerMirroredStrategy with cluster_spec = {'worker': ['10.244.1.86:8899', '10.244.3.64:8899']}, task_type = 'worker', task_id = 1, num_workers = 2, local_devices = ('/job:worker/task:1/device:GPU:0',), communication = CollectiveCommunication.RING


In [4]:
collection_name = db.df_s_2
cursor = collection_name.find()
list_cur = list(cursor)

In [5]:
data = pd.DataFrame(list_cur)
data=data.drop(['_id','Datetime'],axis=1)

In [6]:
data.astype('float')

,s_1도로총교통량,s_1평균통행속도,s_1점유율,s_2도로총교통량,s_2평균통행속도,s_2점유율,s_3도로총교통량,s_3평균통행속도,s_3점유율,s_4도로총교통량,...,s_10도로총교통량,s_10평균통행속도,s_10점유율,s_11도로총교통량,s_11평균통행속도,s_11점유율,s_12도로총교통량,s_12평균통행속도,s_12점유율,휴일
0,210.0,56.000,2.41625,175.0,55.625,3.00375,189.0,53.500,3.84250,216.0,...,259.0,53.3750,3.676250,93.0,62.6250,2.346250,60.0,63.750,1.622500,1.0
1,250.0,54.000,3.04625,183.0,56.375,2.90500,167.0,59.250,2.20875,136.0,...,176.0,55.2500,2.983750,114.0,63.0000,2.616250,59.0,64.375,1.892500,1.0
2,135.0,54.375,2.33750,104.0,59.750,2.03875,83.0,61.625,1.94750,87.0,...,114.0,56.5000,2.622500,64.0,64.7500,2.010000,48.0,66.750,1.752500,1.0
3,83.0,56.750,1.97125,70.0,61.250,1.99000,66.0,61.375,2.23125,77.0,...,69.0,59.5000,2.133750,35.0,65.1250,1.946250,37.0,71.375,1.606250,1.0
4,91.0,60.375,1.85750,80.0,64.500,1.91125,48.0,60.250,1.96250,61.0,...,63.0,60.5000,2.196250,32.0,64.3750,1.765000,47.0,68.000,1.766250,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,230.0,57.750,2.82125,158.0,59.625,2.59625,176.0,56.000,2.90625,197.0,...,214.0,55.3750,3.438750,189.0,65.0000,3.531250,72.0,65.375,2.273750,1.0
4996,157.0,57.000,2.44250,87.0,63.625,2.10125,102.0,57.500,2.72625,128.0,...,144.0,58.0000,2.902500,103.0,64.7500,2.517500,45.0,71.250,1.598750,0.0
4997,88.0,54.500,2.18750,57.0,62.625,1.87875,57.0,60.375,2.01125,76.0,...,86.0,55.3750,2.518750,31.0,67.8750,1.516250,25.0,66.750,1.531250,0.0
4998,46.0,55.875,1.73500,35.0,60.125,1.59250,34.0,64.250,1.72000,34.0,...,26.0,55.5714,2.572857,22.0,71.7143,1.452857,18.0,67.250,1.555000,0.0


In [7]:
# Normalization - 정규화
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scale_clos = ['s_1도로총교통량','s_1평균통행속도','s_1점유율',
             's_2도로총교통량','s_2평균통행속도','s_2점유율',
             's_3도로총교통량','s_3평균통행속도','s_3점유율',
             's_4도로총교통량','s_4평균통행속도','s_4점유율',
             's_5도로총교통량','s_5평균통행속도','s_5점유율',
             's_6도로총교통량','s_6평균통행속도','s_6점유율',
             's_7도로총교통량','s_7평균통행속도','s_7점유율',
             's_8도로총교통량','s_8평균통행속도','s_8점유율',
             's_9도로총교통량','s_9평균통행속도','s_9점유율',
             's_10도로총교통량','s_10평균통행속도','s_10점유율',
             's_11도로총교통량','s_11평균통행속도','s_11점유율',
             's_12도로총교통량','s_12평균통행속도','s_12점유율','휴일']
df_scaled = scaler.fit_transform(data[scale_clos])

df_scaled = pd.DataFrame(data=df_scaled,columns=scale_clos)

train = df_scaled[:1800] # 학습 데이터
test = df_scaled[1800:] # 테스트 데이터

In [8]:
# 학습 시킬 데이터 셋 생성 함수
# 5시간 기반으로 데이터 셋 묶기
def make_dataset(data,label,window_size=5):
    feature_list =[] # 학습
    label_list =[] # 답
    for i in range(len(data)-window_size):
        feature_list.append(np.array(data.iloc[i:i+window_size]))
        label_list.append(np.array(label.iloc[i+window_size]))
    return np.array(feature_list),np.array(label_list)
# 한누리대로 
feature_cols = ['s_1도로총교통량','s_1평균통행속도','s_1점유율',
             's_2도로총교통량','s_2평균통행속도','s_3점유율',
             's_3도로총교통량','s_3평균통행속도','s_4점유율',
             's_4도로총교통량','s_4평균통행속도','s_5점유율',
             's_5도로총교통량','s_5평균통행속도','s_6점유율',
             's_6도로총교통량','s_6평균통행속도','s_7점유율',
             's_7도로총교통량','s_7평균통행속도','s_8점유율',
             's_8도로총교통량','s_8평균통행속도','s_9점유율',
             's_9도로총교통량','s_9평균통행속도','s_10점유율',
             's_10도로총교통량','s_10평균통행속도','s_11점유율',
             's_11도로총교통량','s_11평균통행속도','s_2점유율',
             's_12도로총교통량','s_12평균통행속도','휴일'] # 학습 컬럼
label_cols = ['s_12점유율'] # 답 컬럼

train_feature = train[feature_cols]
train_label = train[label_cols]

# train dataset
train_feature,train_label = make_dataset(train_feature,train_label,5)

# train,test set 생성
X_train,X_test,y_train,y_test = \
train_test_split(train_feature,train_label,test_size=0.8,shuffle=False)

test_feature = test[feature_cols]
test_label = test[label_cols]

# test dataset(실제 예측 데이터)
test_feature,test_label = make_dataset(test_feature,test_label,5)

In [9]:
with multiworker_strategy.scope():
    model = models.Sequential()
    model.add(
        layers.LSTM(32,            # 노드 개수
        input_shape=(X_train.shape[1],X_train.shape[2]),# input_dim 입력 데이터의 피쳐 개수
        activation='relu',
        return_sequences = True)
        )
    model.add(
        layers.LSTM(32,activation='relu')
        )

    model.add(
        layers.Dense(24)
        ) # 예측하고자하는 target이 1시간 ~ 24시간뒤 (output)
    
    model.summary()
    model.compile(loss='mse',optimizer='adam',metrics=['acc'])
callbacks = [ModelCheckpoint(filepath='./chk_point/a.chk', save_freq='epoch')]
model.fit(X_train,y_train,epochs=20,batch_size=32
               ,validation_data=(X_test,y_test)
               ,callbacks=callbacks
               ,shuffle = False
      )

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm (LSTM)                  (None, 5, 32)             8832      
_________________________________________________________________
lstm_1 (LSTM)                (None, 32)                8320      
_________________________________________________________________
dense (Dense)                (None, 24)                792       
Total params: 17,944
Trainable params: 17,944
Non-trainable params: 0
_________________________________________________________________
INFO:tensorflow:Running Distribute Coordinator with mode = 'independent_worker', cluster_spec = {'worker': ['10.244.1.86:8899', '10.244.3.64:8899']}, task_type = 'worker', task_id = 1, environment = None, rpc_layer = 'grpc'
INFO:tensorflow:Using MirroredStrategy with devices ('/job:worker/task:1/device:GPU:0',)
INFO:tensorflow:MultiWorkerMirroredStrategy with cluster_spec = {'wor

In [10]:
model.evaluate(test_feature,test_label,return_dict=True)

INFO:tensorflow:Running Distribute Coordinator with mode = 'independent_worker', cluster_spec = {'worker': ['10.244.1.86:8899', '10.244.3.64:8899']}, task_type = 'worker', task_id = 1, environment = None, rpc_layer = 'grpc'
INFO:tensorflow:Using MirroredStrategy with devices ('/job:worker/task:1/device:GPU:0',)
INFO:tensorflow:MultiWorkerMirroredStrategy with cluster_spec = {'worker': ['10.244.1.86:8899', '10.244.3.64:8899']}, task_type = 'worker', task_id = 1, num_workers = 2, local_devices = ('/job:worker/task:1/device:GPU:0',), communication = CollectiveCommunication.RING
INFO:tensorflow:Using MirroredStrategy with devices ('/job:worker/task:1/device:GPU:0',)
INFO:tensorflow:MultiWorkerMirroredStrategy with cluster_spec = {'worker': ['10.244.1.86:8899', '10.244.3.64:8899']}, task_type = 'worker', task_id = 1, num_workers = 2, local_devices = ('/job:worker/task:1/device:GPU:0',), communication = CollectiveCommunication.RING
100/100 [==============================] - 4s 35ms/step - lo

{'loss': 0.009259577840566635, 'acc': 0.0}

In [11]:
# 예측
pred =model.predict(test_feature)

ValueError: predict is not supported in multi-worker mode.

In [ ]:
# 시각화
plt.figure(figsize=(12,9))
plt.plot(test_label,label='actual') # 실제
plt.plot(pred,label='predicition') # 예측
plt.xlabel('time')
plt.ylabel('share')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()

In [ ]:
# 모델 저장하기
# model.save('s_1.h5')